In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data/raw")
FIG = Path("../reports/figures")

visits = pd.read_csv(DATA / "air_visit_data.csv", parse_dates=["visit_date"])
cal = pd.read_csv(DATA / "date_info.csv", parse_dates=["calendar_date"])
info = pd.read_csv(DATA / "air_store_info.csv")

df = visits.merge(cal, left_on="visit_date", right_on="calendar_date", how="left")
df["lv"] = np.log1p(df["visitors"])
# distance from the store's own average, on the log scale
df["lv_rel"] = df["lv"] - df.groupby("air_store_id")["lv"].transform("mean")
print(df.shape)   # must still be 252,108 rows: a merge must not add rows

(252108, 8)


In [3]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
raw = df.groupby("day_of_week")["visitors"].mean().reindex(order)
rel = df.groupby("day_of_week")["lv_rel"].mean().reindex(order)
print(pd.DataFrame({"raw_mean": raw.round(1), "index_vs_store_avg": np.exp(rel).round(2)}))

             raw_mean  index_vs_store_avg
day_of_week                              
Monday           17.2                0.81
Tuesday          17.7                0.86
Wednesday        19.2                0.94
Thursday         18.9                0.92
Friday           23.1                1.14
Saturday         26.3                1.29
Sunday           23.9                1.08


In [4]:
store_mean = df.groupby("air_store_id")["visitors"].transform("mean")
print(df.assign(store_size=store_mean).groupby("day_of_week")["store_size"].mean().reindex(order).round(1))

day_of_week
Monday       21.3
Tuesday      20.9
Wednesday    20.9
Thursday     20.9
Friday       20.9
Saturday     20.9
Sunday       21.1
Name: store_size, dtype: float64
